# E-Commerce Customer Satisfaction Score Prediction
```


## 1. Setup & Imports


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Resolve project root (notebook lives in notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.ingestion import DataIngestion
from src.data.validation import DataValidation
from src.data.schema import CSAT_SCHEMA, TARGET_COLUMN
from src.data_preprocessing.missing_value_handler import MissingValueHandler
from src.data_preprocessing.feature_engineering import FeatureEngineer
from src.data_preprocessing.preprocessing import DataPreprocessor
from src.models.csat_classifier import CSATClassifier
from src.pipeline.prediction_pipeline import PredictionPipeline

# Load configs
with open(PROJECT_ROOT / "configs/paths.yaml") as f:
    PATHS = yaml.safe_load(f)
with open(PROJECT_ROOT / "configs/training_config.yaml") as f:
    TRAINING_CONFIG = yaml.safe_load(f)

sns.set(style="whitegrid")
print(f"Project root: {PROJECT_ROOT}")


## 2. Dataset Overview

The dataset covers customer satisfaction scores over a one-month period on the **Shopzilla** e-commerce platform.

| Column | Description |
|--------|-------------|
| `Unique id` | Unique record identifier |
| `channel_name` | Service channel (Inbound, Outcall, Email) |
| `category` / `Sub-category` | Interaction type |
| `Customer Remarks` | Free-text feedback (66% missing) |
| `Order_id` | Order identifier |
| `Issue_reported at` / `issue_responded` / `Survey_response_Date` | Timestamps |
| `Customer_City`, `Product_category`, `Item_price`, `connected_handling_time` | High-missing columns (>70%) |
| `Agent_name`, `Supervisor`, `Manager`, `Tenure Bucket`, `Agent Shift` | Agent metadata |
| `CSAT Score` | Target — customer satisfaction (1–5) |


## 3. Data Ingestion & Validation


In [ ]:
ingestion = DataIngestion(config_path=str(PROJECT_ROOT / "configs/paths.yaml"))
validation = DataValidation()

raw_df = ingestion.load_csat_data()
summary = validation.validate_dataset(raw_df)

print(f"Shape: {raw_df.shape}")
print(f"Duplicates: {summary['duplicate_rows']}")
raw_df.head()


In [ ]:
raw_df.info()


## 4. Data Preprocessing


### 4.1 Missing Values

Handled by `src/data_preprocessing/missing_value_handler.py`:
- **Drop** columns with >70% missing: `connected_handling_time`, `Customer_City`, `Product_category`, `Item_price`, `order_date_time`
- **Impute** `Customer Remarks` with `"No Remarks"`
- **Drop rows** with missing `Order_id`


In [ ]:
missing_handler = MissingValueHandler(
    plot_output_path=str(PROJECT_ROOT / PATHS["report_paths"]["missing_values_plot"])
)

missing_counts = missing_handler.get_missing_values(raw_df)
print("Top missing columns:")
print(missing_counts.head(9))

missing_handler.save_missing_values_plot(raw_df)
print(f"Plot saved to: {PATHS['report_paths']['missing_values_plot']}")


In [ ]:
cleaned_df = missing_handler.handle_missing_values(raw_df)
print(f"Shape after missing value handling: {cleaned_df.shape}")
print(f"Remaining nulls: {cleaned_df.isnull().sum().sum()}")


### 4.2 Feature Engineering

Handled by `src/data_preprocessing/feature_engineering.py`:
- Text preprocessing for `Customer Remarks`
- Datetime feature extraction (hour, day, month) from timestamp columns


In [ ]:
feature_engineer = FeatureEngineer()
engineered_df = feature_engineer.engineer_features(raw_df)
print(f"Shape after feature engineering: {engineered_df.shape}")
engineered_df.head()


## 5. Exploratory Data Analysis


In [ ]:
eda_df = engineered_df.copy()

plt.figure(figsize=(10, 6))
order = sorted(eda_df[TARGET_COLUMN].unique())
ax = sns.countplot(x=TARGET_COLUMN, data=eda_df, order=order, palette="viridis")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.title("CSAT Score Distribution", fontsize=14, fontweight="bold")
plt.xlabel("CSAT Score")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
channel_csat = eda_df.groupby("channel_name")[TARGET_COLUMN].mean().sort_values(ascending=False)
print(channel_csat)

plt.figure(figsize=(10, 5))
sns.barplot(x=channel_csat.values, y=channel_csat.index, palette="mako")
plt.title("Average CSAT Score by Channel")
plt.xlabel("Average CSAT Score")
plt.tight_layout()
plt.show()


In [ ]:
low_csat = eda_df[eda_df[TARGET_COLUMN] <= 2]
top_cats = low_csat["category"].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_cats.values, y=top_cats.index, palette="rocket")
plt.title("Top Categories for Low CSAT Scores (1–2)")
plt.xlabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
shift_csat = eda_df.groupby("Agent Shift")[TARGET_COLUMN].mean().sort_values(ascending=False)
print(shift_csat)

plt.figure(figsize=(10, 5))
sns.barplot(x=shift_csat.values, y=shift_csat.index, palette="crest")
plt.title("Average CSAT Score by Agent Shift")
plt.xlabel("Average CSAT Score")
plt.tight_layout()
plt.show()


## 6. Model-Ready Preprocessing


`src/data_preprocessing/preprocessing.py` applies:
- One-hot encoding for 8 categorical columns
- Standard scaling for 9 datetime-derived numeric features

Output: sparse feature matrix ready for the neural network.


In [ ]:
preprocessor = DataPreprocessor()
X, y, fitted_preprocessor = preprocessor.preprocess_csat_data(raw_df)

if hasattr(X, "toarray"):
    X = X.toarray()
X = X.astype("float32")

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")


## 7. Model Training


Architecture from `src/models/csat_classifier.py` (matches original notebook):

```
Dense(128) → Dropout(0.3) → Dense(64) → Dropout(0.3) → Dense(32) → Dropout(0.3) → Dense(5, softmax)
```

Hyperparameters from `configs/training_config.yaml`.


In [ ]:
train_cfg = TRAINING_CONFIG["training"]
model_cfg = TRAINING_CONFIG["model"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=train_cfg["test_size"],
    random_state=train_cfg["random_state"],
    stratify=y,
)

y_train_cat = to_categorical(y_train - 1)
y_test_cat = to_categorical(y_test - 1)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")


In [ ]:
classifier = CSATClassifier(
    input_dim=X_train.shape[1],
    hidden_layers=model_cfg["hidden_layers"],
    dropout_rate=model_cfg["dropout_rate"],
    num_classes=model_cfg["num_classes"],
    optimizer=model_cfg["optimizer"],
    loss=model_cfg["loss"],
)
model = classifier.build_model()
model.summary()


In [ ]:
callbacks = CSATClassifier.get_callbacks(
    early_stopping_config=TRAINING_CONFIG["callbacks"]["early_stopping"],
    reduce_lr_config=TRAINING_CONFIG["callbacks"]["reduce_lr"],
)

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=train_cfg["epochs"],
    batch_size=train_cfg["batch_size"],
    callbacks=callbacks,
    verbose=1,
)


## 7.1 Alternative: Full Training Pipeline


In [ ]:
# One-liner equivalent to main.py train — skips manual steps above
from src.pipeline.training_pipeline import TrainingPipeline

# results = TrainingPipeline(
#     paths_config=str(PROJECT_ROOT / "configs/paths.yaml"),
#     training_config=str(PROJECT_ROOT / "configs/training_config.yaml"),
# ).run()
# print(results)


## 8. Model Evaluation


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()


## 9. Save Artifacts


In [ ]:
import json
import joblib
import os

model_path = PROJECT_ROOT / PATHS["artifact_paths"]["model"]
preprocessor_path = PROJECT_ROOT / PATHS["artifact_paths"]["preprocessor"]
history_path = PROJECT_ROOT / PATHS["artifact_paths"]["training_history"]

for path in (model_path, preprocessor_path, history_path):
    os.makedirs(path.parent, exist_ok=True)

model.save(str(model_path))
joblib.dump(fitted_preprocessor, preprocessor_path)

history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(history_path, "w") as f:
    json.dump(history_dict, f, indent=2)

print(f"Model saved to: {model_path}")
print(f"Preprocessor saved to: {preprocessor_path}")
print(f"Training history saved to: {history_path}")


## 10. Inference Demo

Use `PredictionPipeline` for predictions — same logic used by the FastAPI service and Streamlit app.


In [ ]:
pipeline = PredictionPipeline(paths_config=str(PROJECT_ROOT / "configs/paths.yaml"))
sample = raw_df.head(5)
actual = sample[TARGET_COLUMN].values
predicted = pipeline.predict(sample)

print("Sample predictions (actual → predicted):")
for act, pred in zip(actual, predicted):
    print(f"  {act} → {pred}")


## 11. Next Steps

| Component | Command / Location |
|-----------|-------------------|
| **CLI training** | `python main.py train` |
| **FastAPI server** | `python main.py serve` → http://localhost:8000/docs |
| **Streamlit UI** | `streamlit run streamlit/app.py` |
| **Run tests** | `python test_validation.py`, `python test_preprocessing.py` |

### Key Observations

- CSAT scores are heavily skewed toward 5 (~69% of records)
- Model achieves ~69–70% test accuracy on 5-class classification
- Small train/validation gap suggests reasonable generalization
- `Customer Remarks` free text is excluded from features (handled by `text_preprocessing.py`)
